Only for a CCZ gate, i.e. time is not a NN output but just a hyperparameter that pytorch tunes. 

In [1]:
import torch 
from decimal import Decimal
from schsolve import neural_trainer, schsolver 
import schsolve
import torchdiffeq as tdf 
import cst_n_fn as cfn 
import matplotlib.pyplot as plt 
import const_cczphi as cczphi 
import numpy as np 
import pytorch_warmup as warmup 
from const_cczphi import reduce_r_dim_3q_vector, correction_1q, correction_1q_param

device = "cpu"

print(device)
    
# **** training function below ****
# Note that this has to be coded separately for each use case 
        
def trainer(neural_model, nn_solver_1, nn_solver_2, input_tensor, scheduler, epoch, init_matrix, multiplier = 2.0, print_ = True):
    
    #trial_1q_rot.train()
    neural_model.train()
    nn_solver_1.zero_grad()
    nn_solver_2.zero_grad()
    #nn_solver_3.zero_grad()
    nn_time_output = (neural_model(input_tensor).select(1,0))
    
    pred_outputs_detuning = nn_time_output*(neural_model.range_detuning[1]\
         - neural_model.range_detuning[0]) + neural_model.range_detuning[0]
        
    cczphi.instance.hamiltonian.rabi_tensored["pulse 0"] \
    = cczphi.instance.hamiltonian.rabi_tensored["pulse 1"] \
    = cczphi.instance.hamiltonian.rabi_tensored["pulse 2"]\
    = cfn.const_then_zero_tensor(cfn.rabi, neural_model.gatetime_prediction)

    cczphi.instance.hamiltonian.det_tensored["pulse 0"] \
    = cczphi.instance.hamiltonian.det_tensored["pulse 1"] \
    = cczphi.instance.hamiltonian.det_tensored["pulse 2"] \
    = cczphi.list_to_fn_tensor_var_gatetime(pred_outputs_detuning,\
         neural_model.gatetime_prediction, cczphi.time_steps)    
    
    time_arr_ = torch.linspace(0, 1.0, 3, device = device)*neural_model.gatetime_prediction.max()
    
    # * code below evolves the Hamiltonian with time-dependent controls and computes the state in the subspace of 0s and 1s

    sol_intrm = reduce_r_dim_3q_vector(tdf.odeint(cczphi.instance,\
                                init_matrix, time_arr_, 
                                method = 'dopri5', rtol = 5e-5, atol = 5e-5)[-1], angle_batch = cczphi.angle_batch)

    solution = correction_1q_param(sol_intrm, angle_guess, angle_batch = cczphi.angle_batch) 
    
    #var_rotation_1q = trial_1q_rot().reshape(cczphi.angle_batch, 8, 8)
    #print(sol_intrm.shape)
    #print(torch.angle(sol_intrm[:, 1, 1]))
    #solution = torch.bmm(var_rotation_1q, sol_intrm)

    #solution = torch.exp()
    infidelity_term = cfn.unitary_infidelity_array(solution, \
        cfn.cczp_gate_stack_zzz(input_tensor/neural_model.scale_factor +\
             neural_model.offset), nqbits = 3)

    time_term = multiplier*torch.mean(neural_model.gatetime_prediction)
    
    dict_ = {"time term": time_term.item(), "infidelity term": infidelity_term.item(),
                "gatetime mean.": torch.mean(neural_model.gatetime_prediction)}
    
    loss_instance = infidelity_term + time_term 
    
    neural_model.debug_gradient_time.retain_grad()
    nn_time_output.retain_grad()
    loss_instance.backward()
    
    #nn_solver_3.step()
    nn_solver_2.step()
    nn_solver_1.step()
    
    if scheduler[0] == True:
        scheduler[2].step()
        scheduler[1].step()
    
    if epoch%5 == 0 and print_ == True:  
        
        #print(neural_model.gatetime_prediction.max()*cfn.rabi)
        
        print("Epoch {}: Loss = {:.2E}".format(epoch, Decimal(loss_instance.cpu().detach().numpy().item())))
        #print(trial_1q_rot.f)
        print(dict_)
        print(torch.angle(sol_intrm[:, 1, 1]))
        #print('gradient time!')
        #print(sum(neural_model.debug_gradient_time.grad))
        #print('gradient control!')
    
    return solution.cpu().detach().numpy(), loss_instance.item(), dict_

def scale_and_offset(angle_arr, network):
    
    # * This function scales the angle by some factor (useful for small angles)
    # * And also subtracts an offset s.t. the mean = 0 

    a = (angle_arr - network.offset)*network.scale_factor 

    return a 

/opt/homebrew/Caskroom/miniconda/base/envs/baseClone/lib/python3.11/site-packages/qutip/__init__.py:65: UserWarning: The new version of Cython, (>= 3.0.0) is not supported.
  warnings.warn(


Something to be careful about is that the interaction strength for a multiqubit gate ($N > 2$) now depends on the geometry of arrangement. For $N = 3$ for instance, $V_{dd;1,2} = V_{dd;2,3} = V_{dd;1,3}$ implies a triangle geometry. 